# Study — the sites and the maps (`mppp_studies.sites`)

The Mars 2020 sites of `mppp/data/sites.json` — their sol ranges, waypoints, zooms and official block numbers —
and the figures of the paper that place them on the terrain: the mission traverse with the rover sites, the
cluster zooms, the LMST histograms.  The waypoint table is downloaded (`mppp waypoints --refresh`); the DTM and
orthomosaic tiles come from the mission's released products (`maps` extra: rasterio, pyproj, adjustText).

`make_maps.bat` runs the whole thing on the PC (`mppp maps all`); here the pieces.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/conda: duplicate Intel OpenMP guard; before torch
import json
import logging
from pathlib import Path
import mppp
logging.basicConfig(level=logging.INFO, format="%(message)s")   # the progress lines
print("MPPP", mppp.__version__, "from", Path(mppp.__file__).parent)
from mppp.waypoints.sites import load_site_table
SITES = load_site_table()["sites"]
print(len(SITES), "sites in sites.json")

## 1  Parameters

In [ ]:
OUT = Path(os.environ.get("MPPP_OUTPUTS", "Claude outputs")) / "maps"
BACKGROUND = "ortho"                      # "dtm" or "ortho"
DPI = 150
FIGURES = [2, 3, 6]                       # the paper figures to draw (mppp maps figures ...)

## 2  The sites
Every site: sols, the number of waypoints, the maximum separation, the zooms, the official block.  A site is added
by one line in `sites.json` (or `Settings.sites_extra`); `mppp sites list` prints this table.

In [ ]:
for name, s in sorted(SITES.items()):
    print(f"{name:28s} sols {s['sols']}  wp {s.get('n_waypoints', '?'):>3}  "
          f"sep {s.get('max_waypoint_separation_m', 0):6.1f} m  zcam {s.get('zcam') or '-'}  block {s.get('official_blocks') or '-'}")

## 3  The waypoints
The current waypoint table (the mission's `M20_waypoints` JSON): every end-of-drive position with its site, drive
and sol.  `sync_study_waypoints` keeps the study's copy next to the package snapshot.

In [ ]:
from mppp.waypoints import load_waypoints, snapshot_path
wps = load_waypoints()
print(len(wps), "waypoints; snapshot", snapshot_path())
print(wps[-1])

## 4  The figures — `mppp maps figures 2 3 6 --background ortho`
Figure 2: the traverse and the sites; figure 3: the cluster zooms with the sol numbers; figure 6: the Mars
orthomosaic to the rover sites (E–H the zooms).  `mppp maps zooms` draws every cluster; `mppp maps all` everything.

In [ ]:
import subprocess, sys
cmd = [sys.executable, "-m", "mppp", "maps", "figures", *map(str, FIGURES), "--out", str(OUT), "--dpi", str(DPI), "--background", BACKGROUND]
print(" ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-1000:])
from IPython.display import Image as NbImage, display
for png in sorted(OUT.glob("*.png"))[:3]:
    display(NbImage(filename=str(png)))

## 5  LMST histograms
The local mean solar time of every frame of the consensus blocks, per camera family — the lighting the models
were made under (`mppp_studies.sites.figures`).

In [ ]:
from mppp_studies.sites.figures import lmst_images, lmst_site_row, plot_lmst_histograms
blocks_root = Path(os.environ.get("MPPP_SCAPES", "/scapes"))
if blocks_root.is_dir():
    images, found = lmst_images([blocks_root])                 # {site: {stem: record}} from the newest manifests
    rows = [lmst_site_row(site, list(recs.values())) for site, recs in images.items()]
    print(plot_lmst_histograms(images, rows, OUT / "lmst_histograms.png"))
    display(NbImage(filename=str(OUT / "lmst_histograms.png")))
else:
    print("no blocks root at", blocks_root)